# Bounded T4 inference benchmark
Prepare on CPU, verify all files in Drive, then select one T4. This measures the unchanged 180k model at batch sizes 64, 128 and 256. Each result remains a separate exact model output. It does not train or submit.


In [ ]:
import torch
assert not torch.cuda.is_available(), 'Start this notebook on CPU'
%pip -q install transformers==4.57.6 pyarrow requests


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, hashlib, requests, subprocess, sys
from pathlib import Path
source=Path('/content/hw2-inference-source'); (source/'homeworks/hw2').mkdir(parents=True,exist_ok=True)
commit='eb3f96fb3ba57930ac6dd39f26a48baf4e4ef583'
hashes={'benchmark_inference.py': 'c16c45f49d2e91f954dd69b234d2cc7d4c70f31615c14f7ee756a0048d53a0e0', 'inference_probe_colab.py': '960f9eff672ef766f51e57040c27d15cd8f324248cbed59a28ca511fec2206b3', 'inference.py': '7c10a82dd2001363069dbe5993e303561f153c15afab958540292846b89b7123', 'data.py': 'fcf8884d0547eb54a9d2e8035495dc77e68a22ea3323af4d63029261036fed58', 'checkpoint_io.py': 'd94cf783ec1b5540d5266ddd9817e0d904d3d07a4f97b225215ef4e2c36ee1f1', 'speed.py': '0b65c07a0b40f0553454f60532b26d5964caef6f0859ad56a740e88c338ba5d3'}
for name,digest in hashes.items():
 response=requests.get(f'https://raw.githubusercontent.com/OsVox/hse-nlp/{commit}/homeworks/hw2/{name}',timeout=60); response.raise_for_status()
 assert hashlib.sha256(response.content).hexdigest()==digest
 (source/'homeworks/hw2'/name).write_bytes(response.content)
(source/'source_commit.txt').write_text(commit+'\n')
os.chdir(source)
subprocess.run([sys.executable,'-m','homeworks.hw2.inference_probe_colab'],check=True)


Confirm `INFERENCE_CPU_READY` and export `ready.json` from Drive before switching to T4. The next cell restores only verified bundled files. Do not run the CPU preparation cells again on GPU.


In [ ]:
from google.colab import drive, runtime
try:
 drive.mount('/content/drive')
 from pathlib import Path
 import os, json, hashlib, tarfile, shutil, subprocess, sys
 root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
 prepared=root/'prepared/inference-speed-180k-v1'
 ready=json.loads((prepared/'ready.json').read_text())
 assert ready['cpu_preflight_passed']
 local=Path('/content/hw2-inference-t4'); local.mkdir(exist_ok=False)
 archive=local/'inputs.tar'; shutil.copyfile(prepared/'inputs.tar',archive)
 assert hashlib.sha256(archive.read_bytes()).hexdigest()==ready['inputs_sha256']
 with tarfile.open(archive) as bundle: bundle.extractall(local,filter='data')
 os.chdir(local)
 subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow','requests'],check=True)
 with (prepared/'launcher.log').open('w') as log:
  subprocess.run([sys.executable,'-u','-m','homeworks.hw2.inference_probe_colab','--gpu'],stdout=log,stderr=subprocess.STDOUT,timeout=750,check=True)
finally:
 runtime.unassign()


The worker saves each result to Drive and requests GPU release on success or failure. Its hard timeout is 12 minutes. Inspect `execution.complete.json`, `execution.log` and `benchmarks/inference-speed-180k-t4-v1/results.json` after switching back to CPU. Verify that the runtime is released. The profile pass is excluded from the speed comparison. Compilation is not enabled in this first experiment.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import json, zipfile, hashlib
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
result=root/'benchmarks/inference-speed-180k-t4-v1'
print((result/'results.json').read_text())
archive=result/'export.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for path in sorted(result.iterdir()):
  if path.is_file() and path!=archive: z.write(path,arcname=path.name)
print('EXPORT_SHA256',hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
